# Fine-tune YOLOv8n (SiLU→LeakyReLU) on package-conveyo — for KV260 DPU

รันบน **Google Colab** (Runtime → Change runtime type → **T4 GPU**)

ทำ SiLU→LeakyReLU swap ตอนเทรน (DPU ไม่รองรับ SiLU) แล้ว fine-tune บน dataset `package` 1-class
ผลลัพธ์ = `yolov8n_leaky_pkg_ft.pt` → เอากลับมา quantize+compile สำหรับบอร์ด

**สำคัญ:** slope = 0.1015625 (26/256) ต้องตรงกับ `quantize_yolo_pytorch.py` เป๊ะ

## 1. เช็ค GPU + ติดตั้ง ultralytics 8.4.71

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics==8.4.71 roboflow

## 2. ดึง dataset จาก Roboflow

ใส่ **API key** ของคุณ (Roboflow → Settings → API Key). workspace/project/version เติมให้แล้วตรงกับ export v2

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("guns-workspace-e4rag").project("package-conveyo")
dataset = project.version(2).download("yolov8")
DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml =", DATA_YAML)
!cat {DATA_YAML}

### (ทางเลือก) ถ้าไม่อยากใช้ API key — อัปโหลด zip เอง
ยกเลิก comment แล้วอัปโหลด `package-conveyo.v2i.yolov8.zip` (อยู่ใน `02-dataset/raw/`)

In [ ]:
# from google.colab import files
# up = files.upload()   # เลือก package-conveyo.v2i.yolov8.zip
# !unzip -q -o package-conveyo.v2i.yolov8.zip -d /content/pkg
# DATA_YAML = "/content/pkg/data.yaml"

## 3. Fine-tune พร้อม SiLU→LeakyReLU swap callback

In [ ]:
import torch.nn as nn
from ultralytics import YOLO

SLOPE = 0.1015625  # 26/256 — ต้องตรงกับ quantize_yolo_pytorch.py

def swap_silu_to_leaky(module):
    n = 0
    for name, child in module.named_children():
        if isinstance(child, nn.SiLU):
            setattr(module, name, nn.LeakyReLU(SLOPE, inplace=False))
            n += 1
        else:
            n += swap_silu_to_leaky(child)
    return n

def cb(trainer):
    n = swap_silu_to_leaky(trainer.model)
    m = 0
    if getattr(trainer, 'ema', None) is not None and getattr(trainer.ema, 'ema', None) is not None:
        m = swap_silu_to_leaky(trainer.ema.ema)
    print(f'[cb] SiLU->LeakyReLU({SLOPE}) swapped: model={n} ema={m}')
    if n == 0:
        raise SystemExit('[cb] FATAL: no SiLU found')

model = YOLO('yolov8n.pt')
model.add_callback('on_pretrain_routine_end', cb)

results = model.train(
    data=DATA_YAML, epochs=15, imgsz=640, batch=16, device=0,
    optimizer='SGD', lr0=0.001, warmup_epochs=1.0,
    pretrained=False, plots=True, val=True, cache=True,
    project='runs_ft', name='yolov8n_leaky_pkg',
)

## 4. ตรวจผล + ดาวน์โหลด best.pt กลับเครื่อง

In [ ]:
import shutil, glob, os
best = 'runs_ft/yolov8n_leaky_pkg/weights/best.pt'
assert os.path.exists(best), best
out = 'yolov8n_leaky_pkg_ft.pt'
shutil.copy(best, out)
print('mAP results dir:'); 
print(open('runs_ft/yolov8n_leaky_pkg/results.csv').read().splitlines()[-1])
from google.colab import files
files.download(out)

## 5. (แนะนำ) verify ว่า activation เป็น LeakyReLU จริง ก่อนเอาไป quantize

In [ ]:
import torch.nn as nn
from ultralytics import YOLO
m = YOLO('yolov8n_leaky_pkg_ft.pt')
silu = sum(isinstance(x, nn.SiLU) for x in m.model.modules())
leaky = sum(isinstance(x, nn.LeakyReLU) for x in m.model.modules())
print(f'SiLU={silu}  LeakyReLU={leaky}  (ควรได้ SiLU=0)')